In [8]:
import os
from elasticsearch import Elasticsearch
from openai import OpenAI

ES_API_KEY = os.getenv("ES_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

es_client = (
    Elasticsearch(
        "https://98b77db70f624d4ba658e9e2cb1b55d6.us-central1.gcp.cloud.es.io:443",
        api_key=ES_API_KEY,
    )
    if ES_API_KEY
    else None
)

openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

index_source_fields = {
    "search-avw2": [
        "text"
    ]
}
def get_elasticsearch_results(query):
    if es_client is None:
        raise RuntimeError("ES_API_KEY is not set.")
    es_query = {
        "retriever": {
            "standard": {
                "query": {
                    "multi_match": {
                        "query": query,
                        "fields": [
                            "text"
                        ]
                    }
                }
            }
        },
        "size": 3
    }
    result = es_client.search(index="search-avw2", body=es_query)
    return result["hits"]["hits"]
def create_openai_prompt(results):
    context = ""
    for hit in results:
        ## For semantic_text matches, we need to extract the text from the highlighted field
        if "highlight" in hit:
            highlighted_texts = []
            for values in hit["highlight"].values():
                highlighted_texts.extend(values)
            context += "\n --- \n".join(highlighted_texts)
        else:
            context_fields = index_source_fields.get(hit["_index"])
            for source_field in context_fields:
                hit_context = hit["_source"][source_field]
                if hit_context:
                    context += f"{source_field}: {hit_context}\n"
    prompt = f"""
  Instructions:
  
  - You are an assistant for question-answering tasks.
  - Answer questions truthfully and factually using only the context presented.
  - If you don't know the answer, just say that you don't know, don't make up an answer.
  - You must always cite the document where the answer was extracted using inline academic citation style [], using the position.
  - Use markdown format for code examples.
  - You are correct, factual, precise, and reliable.
  
  Context:
  {context}
  
  """
    return prompt
def generate_openai_completion(user_prompt, question):
    if openai_client is None:
        raise RuntimeError("OPENAI_API_KEY is not set.")
    response = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": user_prompt},
            {"role": "user", "content": question},
        ]
    )
    return response.choices[0].message.content
if __name__ == "__main__":
    question = "my question"
    if es_client is None or openai_client is None:
        print("Set ES_API_KEY and OPENAI_API_KEY to run the example.")
    else:
        elasticsearch_results = get_elasticsearch_results(question)
        context_prompt = create_openai_prompt(elasticsearch_results)
        try:
            openai_completion = generate_openai_completion(context_prompt, question)
            print(openai_completion)
        except Exception as exc:
            print(f"OpenAI request failed: {exc}")

OpenAI request failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [1]:
%pip install -qU elasticsearch openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from elasticsearch import Elasticsearch
from openai import OpenAI